# BASTION: Simulated Data Example

This notebook demonstrates **pybastion** on a controlled simulation where the
true data-generating process (DGP) is known.  It reproduces the core results
from Section 5 of:

> Cho, J. B. & Matteson, D. S. (2026). *BASTION: A Bayesian Framework for
> Trend and Seasonality Decomposition.* arXiv:2601.18052.

## What is BASTION?

BASTION (Bayesian Adaptive Seasonality and Trend DecompositION) decomposes a
univariate time series into:

- **Trend** — locally smooth, piecewise-linear; estimated with a global-local
  horseshoe prior on second differences.
- **Seasonal components** — one per specified period *K*.
- **Outliers** (optional) — sparse additive spikes (horseshoe+ prior).
- **Remainder** — Gaussian noise with constant or stochastic volatility.

Posterior inference uses a Gibbs sampler with O(N) sparse-precision updates
(Rue 2001).

## This example

We generate a length-500 series with a piecewise-linear trend, two
seasonal components (periods 7 and 30), three point outliers, and i.i.d.
Gaussian noise.  We then fit BASTION, inspect the decomposition, and compare
estimated components to the ground truth via MSE and coverage — including the
1 000-replicate benchmarks from Tables 2–3 of the paper.

In [ ]:
import os, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
PROJECT_DIR  = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
OUTPUT_DIR   = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

sys.path.insert(0, PROJECT_DIR)
from pybastion import fit_BASTION

# ── QUICK_MODE ────────────────────────────────────────────────────
# True  → ~1-3 min   (short chain, noisier posteriors).
# False → ~30–60 min (publication-quality).
QUICK_MODE = True

if QUICK_MODE:
    nsave, nburn, nskip, nchains = 50, 50, 0, 1
else:
    nsave, nburn, nskip, nchains = 2000, 5000, 4, 2

nstot = nburn + (nskip + 1) * nsave
print(f"pybastion loaded from: {PROJECT_DIR}")
print(f"QUICK_MODE={QUICK_MODE}: {nstot:,} total MCMC steps × {nchains} chain(s)")
print(f"  nsave={nsave}, nburn={nburn}, nskip={nskip}, nchains={nchains}")

## 1. Generate the Simulated Series

The DGP (DGP 1 in the paper, Table 1) uses:
- Piecewise-linear trend with random slopes drawn from U(−20, 20) × 0.04.
- Two Fourier-pair seasonal components: S₇ and S₃₀.
- Additive Gaussian noise with σ = 1.
- Three point outliers with magnitude ±U(15, 25).

In [ ]:
def generate_T(n=500, seed=4):
    rng = np.random.default_rng(seed)
    slopes = rng.uniform(-20, 20, 4) * 0.04
    bp = [0, 125, 250, 375, 500]
    T = np.zeros(n)
    for i in range(4):
        seg = np.arange(bp[i], bp[i + 1])
        T[seg] = T[bp[i]] + slopes[i] * np.arange(len(seg))
    return T

def gen_sim(n=500, seed=4):
    rng = np.random.default_rng(seed)
    t   = np.arange(1, n + 1)
    T   = generate_T(n, seed=seed)
    S7  = 4 * np.sin(2 * np.pi * t / 7)  + 4 * np.cos(2 * np.pi * t / 7)
    S30 = 5 * np.sin(2 * np.pi * t / 30) + 5 * np.cos(2 * np.pi * t / 30)
    eps = rng.normal(0, 1, n)
    y   = T + S7 + S30 + eps
    out_idx = [50, 200, 400]
    for i in out_idx:
        y[i] += rng.choice([-1, 1]) * rng.uniform(15, 25)
    return {"y": y, "T": T, "S7": S7, "S30": S30, "outliers": out_idx}

sim = gen_sim(n=500, seed=4)
t   = np.arange(1, 501)

fig, ax = plt.subplots(figsize=(10, 3))
ax.scatter(t, sim["y"], s=6, color="black", alpha=0.6, label="Observed y")
ax.plot(t, sim["T"] + sim["S7"] + sim["S30"],
        linewidth=1.2, color="steelblue", label="True signal (T+S)")
ax.plot(t, sim["T"], linewidth=1.2, color="navy", linestyle="--",
        label="True trend T")
for i in sim["outliers"]:
    ax.axvline(i + 1, color="red", linestyle=":", linewidth=1.0, alpha=0.7)
ax.set_xlabel("Time step"); ax.set_ylabel("Value")
ax.set_title("Simulated series — true components shown")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()
print(f'N = {len(sim["y"])}, outlier positions: {sim["outliers"]}')

## 2. Fit BASTION

| Parameter | Quick | Full | Meaning |
|-----------|-------|------|---------|
| `Ks` | `[7, 30]` | `[7, 30]` | Seasonal periods |
| `Outlier` | `True` | `True` | Horseshoe+ outlier component |
| `nsave` | 50 | 2 000 | Saved draws |
| `nburn` | 50 | 5 000 | Burn-in |
| `nskip` | 0 | 4 | Thinning interval |
| `nchains` | 1 | 2 | Independent chains |

Total MCMC steps per chain: `nburn + (nskip+1) × nsave`.
Toggle `QUICK_MODE` in the imports cell to switch.

In [ ]:
result  = fit_BASTION(
    sim["y"],
    Ks=[7, 30],
    Outlier=True,
    cl=0.95,
    obsSV="const",
    nsave=nsave, nburn=nburn, nskip=nskip, nchains=nchains,
    seed=40,
)
summary = result["summary"]
print("Components:", [k for k in summary if k.endswith("_sum")])

## 3. Signal and Trend

`Signal` = Trend + all seasonal components (joint posterior, not summed CIs).
Red dotted lines mark the true outlier positions.

In [ ]:
trend_mean  = np.asarray(summary["Trend_sum"]["Mean"]).ravel()
trend_lo    = np.asarray(summary["Trend_sum"]["CR_lower"]).ravel()
trend_hi    = np.asarray(summary["Trend_sum"]["CR_upper"]).ravel()
signal_mean = np.asarray(summary["Signal_sum"]["Mean"]).ravel()
signal_lo   = np.asarray(summary["Signal_sum"]["CR_lower"]).ravel()
signal_hi   = np.asarray(summary["Signal_sum"]["CR_upper"]).ravel()
s7_mean     = np.asarray(summary["Seasonal7_sum"]["Mean"]).ravel()
s30_mean    = np.asarray(summary["Seasonal30_sum"]["Mean"]).ravel()
s7_lo       = np.asarray(summary["Seasonal7_sum"]["CR_lower"]).ravel()
s7_hi       = np.asarray(summary["Seasonal7_sum"]["CR_upper"]).ravel()
s30_lo      = np.asarray(summary["Seasonal30_sum"]["CR_lower"]).ravel()
s30_hi      = np.asarray(summary["Seasonal30_sum"]["CR_upper"]).ravel()

n = len(sim["y"]); t = np.arange(1, n + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(t, sim["y"], s=6, color="black", alpha=0.5, zorder=1, label="Observed")
ax.fill_between(t, signal_lo, signal_hi, color="grey", alpha=0.3, zorder=2,
                label="95 % credible band")
ax.plot(t, signal_mean, lw=1.5, color="black", zorder=3, label="Signal (T+S)")
ax.plot(t, trend_mean,  lw=1.5, color="red",   zorder=4, label="Trend")
for i in sim["outliers"]:
    ax.axvline(i + 1, color="red", ls="--", lw=1.0, alpha=0.6)
ax.set_xlabel("Time step"); ax.set_ylabel("Value")
ax.set_title("BASTION: estimated signal and trend")
ax.legend(fontsize=9); fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "sim_signal.pdf"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Trend Recovery

The horseshoe prior adapts sharply at breakpoints while staying smooth elsewhere.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.fill_between(t, trend_lo, trend_hi, color="grey", alpha=0.35,
                label="95 % credible band")
ax.plot(t, trend_mean, lw=1.5, color="black", label="pybastion posterior mean")
ax.plot(t, sim["T"],   lw=1.5, color="steelblue", ls="--", label="True trend")
ax.set_xlabel("Time step"); ax.set_ylabel("Value")
ax.set_title("Trend recovery: pybastion vs truth")
ax.legend(fontsize=9); fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "sim_trend.pdf"), dpi=150, bbox_inches="tight")
plt.show()

mse_trend  = float(np.mean((trend_mean  - sim["T"]) ** 2))
mse_signal = float(np.mean((signal_mean - (sim["T"] + sim["S7"] + sim["S30"])) ** 2))
print(f"This run — Trend MSE : {mse_trend:.3f}")
print(f"This run — Signal MSE: {mse_signal:.3f}")

## 5. Seasonal Components

Periods 7 and 30 with 95 % credible bands compared against the truth.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, mean, lo, hi, true_, label in [
    (axes[0], s7_mean,  s7_lo,  s7_hi,  sim["S7"],  "Period-7 seasonality"),
    (axes[1], s30_mean, s30_lo, s30_hi, sim["S30"], "Period-30 seasonality"),
]:
    ax.fill_between(t, lo, hi, color="grey", alpha=0.35, label="95 % CI")
    ax.plot(t, mean,  lw=1.2, color="black",     label="pybastion")
    ax.plot(t, true_, lw=1.2, color="steelblue", ls="--", label="True seasonal")
    ax.set_ylabel("Value"); ax.set_title(label); ax.legend(fontsize=8)
axes[-1].set_xlabel("Time step")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "sim_seasonality.pdf"), dpi=150,
            bbox_inches="tight")
plt.show()

## 6. MSE Table (Paper Table 2, DGP 1)

TBATS / MSTL / STR values are 1 000-replicate averages from the paper.
**R BASTION (paper)** is the paper's reported average.
**pybastion (this run)** is computed from the single fit above.

In [ ]:
mse_season = float(np.mean(
    ((s7_mean + s30_mean) - (sim["S7"] + sim["S30"])) ** 2
))

table2 = pd.DataFrame(
    {
        "Signal MSE":   [13.878, 11.760, 10.829, 0.376, round(mse_signal, 3)],
        "Trend MSE":    [ 3.624, 13.227, 13.785, 0.581, round(mse_trend,  3)],
        "Seasonal MSE": [11.672,  2.712,  2.575, 0.536, round(mse_season, 3)],
    },
    index=["TBATS", "MSTL", "STR",
           "R BASTION (paper avg)", "pybastion (this run)"],
)
print("Table 2 — DGP 1 MSE (lower is better)")
print(table2.to_string())
print()
print("Paper values: 1 000-replicate averages of R BASTION (Cho & Matteson 2026).")
print('"This run": single realisation fitted by pybastion.')
if QUICK_MODE:
    print("(QUICK_MODE — results improve substantially with the full chain.)")

## 7. Coverage Table (Paper Table 3, DGP 1)

**R BASTION (paper)** is the paper's 1 000-replicate average coverage.
**pybastion (this run)** is the fraction of time steps where the true value
falls inside the 95 % posterior credible interval (single-replicate analogue).

In [ ]:
true_signal = sim["T"] + sim["S7"] + sim["S30"]
cov_signal = float(np.mean((signal_lo <= true_signal) & (true_signal <= signal_hi)))
cov_trend  = float(np.mean((trend_lo  <= sim["T"])    & (sim["T"]    <= trend_hi)))
cov_s7     = float(np.mean((s7_lo     <= sim["S7"])   & (sim["S7"]   <= s7_hi)))
cov_s30    = float(np.mean((s30_lo    <= sim["S30"])  & (sim["S30"]  <= s30_hi)))

table3 = pd.DataFrame(
    {
        "STR (paper)":           [0.799, 0.616, 0.815, "—"],
        "R BASTION (paper avg)": [0.998, 0.970, 0.999, "—"],
        "pybastion (this run)":  [
            round(cov_signal, 3), round(cov_trend, 3),
            round(cov_s7, 3),     round(cov_s30, 3),
        ],
    },
    index=["Signal", "Trend", "Seasonal (K=7)", "Seasonal (K=30)"],
)
print("Table 3 — DGP 1 — 95 % coverage (higher is better)")
print(table3.to_string())
print()
print("Paper values: 1 000-replicate averages. Nominal level is 95 %.")
print('"This run": time-average pointwise coverage from a single replicate.')
if QUICK_MODE:
    print("(QUICK_MODE — coverage improves substantially with the full chain.)")